# Financial Market Dashboard
## Interactive Visualization Tool

This notebook creates an interactive dashboard for analyzing key financial market indicators including:
- Inflation metrics (PCE, Core PCE, CPI)
- Treasury yields (2Y, 10Y, 30Y)
- Liquidity measures (Fed Balance Sheet, M2 Money Supply)
- S&P 500 market performance

## 1. Initial Setup

First, we'll install and import all required packages:

In [14]:
# Install required packages
!pip install panel hvplot pandas yfinance fredapi

# Import libraries
import panel as pn
import hvplot.pandas
import pandas as pd
import yfinance as yf
from fredapi import Fred
from datetime import datetime, timedelta

# Initialize Panel
pn.extension()

Defaulting to user installation because normal site-packages is not writeable


## 2. Data Loading

This section loads market data from FRED and Yahoo Finance with robust error handling:

In [15]:
def load_market_data():
    """Load market data with consistent timezone handling"""
    try:
        fred = Fred(api_key='ffa1c3fabd429889449e01d7e3cd1c5f')
        
        # Set date range (timezone-naive)
        end_date = datetime.now()
        start_date = end_date - timedelta(days=5*365)
        
        # Initialize empty data dictionary
        data = {}
        
        # FRED series configuration
        fred_series = {
            'PCE': 'PCE',
            'Core PCE': 'PCEPILFE',
            'CPI': 'CPIAUCSL',
            '2Y Yield': 'DGS2',
            '10Y Yield': 'DGS10',
            '30Y Yield': 'DGS30',
            'Fed Balance Sheet': 'WALCL',
            'M2 Money Supply': 'M2SL'
        }
        
        # Load FRED data (ensure timezone-naive)
        for name, code in fred_series.items():
            try:
                series = fred.get_series(code, start_date, end_date)
                if not series.empty:
                    series.index = pd.to_datetime(series.index).tz_localize(None)
                    data[name] = series
            except Exception as e:
                print(f"Error loading {name}: {str(e)}")
        
        # Load S&P 500 data (convert to timezone-naive)
        try:
            sp500_data = yf.Ticker("^GSPC").history(
                start=start_date.strftime('%Y-%m-%d'),
                end=end_date.strftime('%Y-%m-%d')
            )
            
            if not sp500_data.empty:
                sp500_data.index = sp500_data.index.tz_localize(None)
                for col in ['Close', 'Adj Close', 'Price']:
                    if col in sp500_data.columns:
                        data['SP500'] = sp500_data[col]
                        break
        except Exception as e:
            print(f"Error loading S&P 500: {str(e)}")
        
        # Create DataFrame with consistent timezones
        df = pd.DataFrame({
            k: v for k, v in data.items()
            if v is not None and not v.empty
        }).dropna(how='all')
        
        if df.empty:
            raise ValueError("No data was successfully loaded")
            
        print(f"Successfully loaded data with columns: {list(df.columns)}")
        return df
        
    except Exception as e:
        print(f"Critical error in data loading: {str(e)}")
        return pd.DataFrame()

# Load the data
market_data = load_market_data()

Successfully loaded data with columns: ['PCE', 'Core PCE', 'CPI', '2Y Yield', '10Y Yield', '30Y Yield', 'Fed Balance Sheet', 'M2 Money Supply', 'SP500']


## 3. Dashboard Controls

Create interactive widgets for data exploration:

In [16]:
# Check available columns
available_columns = market_data.columns.tolist()

# Create widgets
series_selector = pn.widgets.MultiChoice(
    name='Select Data Series',
    options=available_columns,
    value=['SP500'] if 'SP500' in available_columns else available_columns[:1],
    width=300
)

date_range_slider = pn.widgets.DateRangeSlider(
    name='Date Range',
    start=market_data.index.min(),
    end=market_data.index.max(),
    value=(market_data.index.min(), market_data.index.max()),
    width=500
)

# Transformation controls
normalize_toggle = pn.widgets.Checkbox(name='Normalize to 100', value=False)
invert_selector = pn.widgets.MultiChoice(name='Invert Series', options=[], width=200)
log_toggle = pn.widgets.Checkbox(name='Log Scale', value=False)

# Update invert options when series selection changes
def update_invert_options(event):
    invert_selector.options = event.new
    
series_selector.param.watch(update_invert_options, 'value')

Watcher(inst=MultiChoice(name='Select Data Series', options=['PCE', 'Core PCE', ...], value=['SP500']), cls=<class 'panel.widgets.select.MultiChoice'>, fn=<function update_invert_options at 0x1127a3940>, mode='args', onlychanged=True, parameter_names=('value',), what='value', queued=False, precedence=0)

## 4. Interactive Visualization

Create the main plot that responds to control selections:

In [17]:
@pn.depends(
    series_selector.param.value,
    date_range_slider.param.value,
    normalize_toggle.param.value,
    invert_selector.param.value,
    log_toggle.param.value
)
def create_interactive_plot(series, date_range, normalize, invert_series, log_scale):
    # Filter data
    plot_df = market_data.loc[date_range[0]:date_range[1], series].copy()
    
    # Apply transformations
    if normalize:
        plot_df = plot_df / plot_df.iloc[0] * 100
        
    for col in invert_series:
        if col in plot_df.columns:
            plot_df[col] = -plot_df[col]
    
    # Create plot
    return plot_df.hvplot(
        width=900,
        height=500,
        grid=True,
        logy=log_scale,
        yformatter="%.0f",
        legend='top',
        title="Financial Market Dashboard"
    )

## 5. Dashboard Assembly

Combine all components into the final dashboard:

In [19]:
# Initialize Panel with ipywidgets
pn.extension(comms='ipywidgets')

# Create dashboard layout
dashboard = pn.Row(
    pn.WidgetBox(
        "## Dashboard Controls",
        series_selector,
        date_range_slider,
        pn.Row(normalize_toggle, log_toggle),
        invert_selector,
        width=350
    ),
    create_interactive_plot,
    sizing_mode='stretch_width'
)

# Display with fallback
try:
    display(dashboard)
except Exception as e:
    print(f"Display error: {str(e)}")
    print("Trying alternative display method...")
    dashboard.show()

BokehModel(combine_events=True, render_bundle={'docs_json': {'e96dff13-eadf-4267-814b-cb5e1800a1a8': {'version…

UnknownReferenceError: can't resolve reference '06ec180b-6c6a-4e44-99e4-ccc9469be311'

UnknownReferenceError: can't resolve reference '06ec180b-6c6a-4e44-99e4-ccc9469be311'

UnknownReferenceError: can't resolve reference '06ec180b-6c6a-4e44-99e4-ccc9469be311'

UnknownReferenceError: can't resolve reference '06ec180b-6c6a-4e44-99e4-ccc9469be311'

...

SkipRendering: bokeh backend could not plot any Elements in the Overlay.

UnknownReferenceError: can't resolve reference '25504686-a798-43a7-b7e5-aebd8aa1036c'

UnknownReferenceError: can't resolve reference '25504686-a798-43a7-b7e5-aebd8aa1036c'

UnknownReferenceError: can't resolve reference '25504686-a798-43a7-b7e5-aebd8aa1036c'

UnknownReferenceError: can't resolve reference '25504686-a798-43a7-b7e5-aebd8aa1036c'